# Exercício — Ciclo de Vida dos Dados com CRISP-DM

## Censo Demográfico 2022 + API REST do IBGE + KNN

Neste exercício vamos aplicar as etapas do **CRISP-DM** em um pequeno projeto de Ciência de Dados.

Objetivo geral:

> Usar dados do **Censo Demográfico 2022**, obtidos pela API REST do IBGE, para selecionar **100 municípios do Estado de São Paulo** e classificá-los em grupos de porte populacional com o algoritmo **K-Nearest Neighbors (KNN)**.

As etapas do exercício serão:

1. Compreensão do Negócio
2. Compreensão dos Dados
3. Preparação dos Dados
4. Modelagem
5. Avaliação
6. Implantação


# 1. Compreensão do Negócio

A população urbana dos municípios brasileiros apresenta grande variação.

Neste exercício queremos construir um modelo didático capaz de classificar municípios em três grupos de porte populacional:

- **Pequena**
- **Média**
- **Grande**

## Pergunta do projeto

É possível usar a população residente urbana de um município para classificá-lo em um grupo de porte populacional utilizando KNN?

## Objetivo

1. obter dados do Censo Demográfico 2022 pela API REST do IBGE;
2. carregar os municípios do Estado de São Paulo;
3. selecionar 100 municípios;
4. construir um `DataFrame`;
5. criar classes de porte populacional;
6. separar dados de treino e teste;
7. treinar um modelo KNN;
8. avaliar o modelo;
9. usar o modelo para classificar uma nova observação.


# 2. Compreensão dos Dados

A fonte utilizada será a **API de Dados Agregados do IBGE**, que alimenta o SIDRA.

Para o Censo Demográfico 2022 utilizaremos a **Tabela SIDRA 9923 — População residente, por situação do domicílio**.

> **Correção importante:** a Tabela SIDRA 202, utilizada na versão anterior do notebook, contém censos até 2010 e não deve ser usada para obter os valores de 2022.

Neste exercício utilizaremos:

- agregado (tabela): **`9923`**;
- período: **`2022`**;
- variável: **`93`** — População residente;
- nível territorial: **`N6`** — Município;
- UF: **`N3[35]`** — Estado de São Paulo;
- situação do domicílio: **urbana**.


## 2.1 Importar bibliotecas


In [7]:
import requests
import pandas as pd

pd.options.display.float_format = "{:,.2f}".format


## 2.2 Consultar a API do IBGE

Em vez de realizar 100 requisições separadas, vamos solicitar em uma única consulta os municípios pertencentes ao Estado de São Paulo.

A expressão:

`N6[N3[35]]`

significa:

> todos os municípios (`N6`) pertencentes ao Estado de São Paulo (`N3[35]`).


In [8]:
url = (
    "https://servicodados.ibge.gov.br/api/v3/agregados/9923/"
    "periodos/2022/variaveis/93"
    "?localidades=N6[N3[35]]"
    "&classificacao=1[1]"
)

resposta = requests.get(url, timeout=60)
resposta.raise_for_status()

print("Status HTTP:", resposta.status_code)


Status HTTP: 200


## 2.3 Converter a resposta para JSON


In [9]:
dados = resposta.json()

type(dados), len(dados)


(list, 1)

## 2.4 Investigar a estrutura interna do JSON

Antes de transformar os dados em tabela, observe os níveis existentes na resposta da API.


In [10]:
dados[0].keys()


dict_keys(['id', 'variavel', 'unidade', 'resultados'])

In [5]:
dados[0]["variavel"], dados[0]["unidade"]


('População residente', 'Pessoas')

In [6]:
dados[0]["resultados"][0].keys()


dict_keys(['classificacoes', 'series'])

### Atividade 1

Investigue o JSON retornado pela API e identifique:

1. o nome da variável;
2. a unidade de medida;
3. onde está o nome do município;
4. onde está o código do município;
5. onde está o período;
6. onde está armazenado o valor da população urbana.


# 3. Preparação dos Dados

Agora vamos transformar o JSON em uma estrutura tabular.

Cada município será uma **observação**.

A variável que utilizaremos será:

`populacao_urbana`


## 3.1 Extrair os municípios e suas populações


In [11]:
resultados = []

for resultado in dados[0]["resultados"]:
    for serie in resultado["series"]:

        localidade = serie["localidade"]
        valores = serie["serie"]

        valor_2022 = valores.get("2022")

        if valor_2022 not in (None, "-", "...", "X"):
            resultados.append({
                "codigo": int(localidade["id"]),
                "municipio": localidade["nome"],
                "ano": 2022,
                "populacao_urbana": int(valor_2022)
            })

len(resultados)


645

### Atividade 2

- Explique o que o código anterior fez.

- Observe especialmente esta sequência:

`dados → resultados → series → localidade → serie`

Por que foi necessário percorrer vários níveis do JSON?


## 3.2 Construir o DataFrame


In [12]:
df_sp = pd.DataFrame(resultados)

df_sp.head()


,codigo,municipio,ano,populacao_urbana
0,3500105,Adamantina - SP,2022,33536
1,3500204,Adolfo - SP,2022,4005
2,3500303,Aguaí - SP,2022,29895
3,3500402,Águas da Prata - SP,2022,6459
4,3500501,Águas de Lindóia - SP,2022,15627


In [13]:
print("Quantidade de municípios recuperados:", len(df_sp))


Quantidade de municípios recuperados: 645


## 3.3 Selecionar 100 municípios

Vamos escolher **100 municípios aleatoriamente**, usando `random_state=42`.

O valor fixo de `random_state` garante que todos os alunos obtenham a mesma amostra.


In [14]:
df = (
    df_sp
    .sample(n=100, random_state=42)
    .sort_values("populacao_urbana")
    .reset_index(drop=True)
)

df


,codigo,municipio,ano,populacao_urbana
0,3507209,Borá - SP,2022,738
1,3515806,Flora Rica - SP,2022,1216
2,3555307,Turmalina - SP,2022,1261
3,3532843,Nova Canaã Paulista - SP,2022,1352
4,3554755,Trabiju - SP,2022,1554
...,...,...,...,...
95,3515004,Embu das Artes - SP,2022,249223
96,3518701,Guarujá - SP,2022,286839
97,3551009,São Vicente - SP,2022,329607
98,3549904,São José dos Campos - SP,2022,688155


## 3.4 Verificar tipos e dados ausentes


In [15]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   codigo            100 non-null    int64 
 1   municipio         100 non-null    object
 2   ano               100 non-null    int64 
 3   populacao_urbana  100 non-null    int64 
dtypes: int64(3), object(1)
memory usage: 3.3+ KB


In [16]:
df.isnull().sum()


,0
codigo,0
municipio,0
ano,0
populacao_urbana,0


## 3.5 Estatística descritiva


In [17]:
df["populacao_urbana"].describe().round(2)


,populacao_urbana
count,100.00
mean,"55,859.13"
std,"141,198.55"
min,738.00
25%,"5,723.25"
50%,"13,058.50"
75%,"42,254.25"
max,"1,132,934.00"


## 3.6 Criar classes de porte populacional

Como esta é uma atividade didática, vamos dividir os 100 municípios em três grupos de tamanhos aproximadamente semelhantes usando os tercis da própria amostra.

Assim teremos classes mais equilibradas para o treinamento do KNN.

> Essas classes são criadas apenas para fins didáticos e não representam uma classificação oficial do IBGE.


In [18]:
df["porte"] = pd.qcut(
    df["populacao_urbana"],
    q=3,
    labels=["Pequena", "Média", "Grande"]
)

df


,codigo,municipio,ano,populacao_urbana,porte
0,3507209,Borá - SP,2022,738,Pequena
1,3515806,Flora Rica - SP,2022,1216,Pequena
2,3555307,Turmalina - SP,2022,1261,Pequena
3,3532843,Nova Canaã Paulista - SP,2022,1352,Pequena
4,3554755,Trabiju - SP,2022,1554,Pequena
...,...,...,...,...,...
95,3515004,Embu das Artes - SP,2022,249223,Grande
96,3518701,Guarujá - SP,2022,286839,Grande
97,3551009,São Vicente - SP,2022,329607,Grande
98,3549904,São José dos Campos - SP,2022,688155,Grande


### Atividade 3 — Quantos municípios existem em cada classe?


In [19]:
df["porte"].value_counts().sort_index()


,count
porte,
Pequena,34
Média,32
Grande,34


# 4. Modelagem

Utilizaremos o algoritmo **K-Nearest Neighbors — KNN**.

Neste exercício:

- Entrada `X` = população urbana;
- Saída   `y` = porte populacional do município.

Caso seja inserida uma nova cidade no dataset, um modelo tipo KNN verificará quais observações conhecidas estão mais próximas de uma nova cidade.

> Neste exemplo há apenas uma variável de entrada. O objetivo principal é **praticar o ciclo CRISP-DM** e o verificar o funcionamento de treino/teste com KNN.


## 4.1 Definir X e y


In [20]:
X = df[["populacao_urbana"]]
y = df["porte"]


In [21]:
X.head()


,populacao_urbana
0,738
1,1216
2,1261
3,1352
4,1554


In [22]:
y.head()


,porte
0,Pequena
1,Pequena
2,Pequena
3,Pequena
4,Pequena


## 4.2 Separar treino e teste

Vamos utilizar:

- 70% dos municípios para treino;
- 30% para teste.

O argumento `stratify=y` preserva aproximadamente a proporção das três classes nos dois conjuntos.


In [23]:
from sklearn.model_selection import train_test_split

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Treino:", len(X_treino))
print("Teste :", len(X_teste))


Treino: 70
Teste : 30


## 4.3 Criar o modelo KNN


In [24]:
from sklearn.neighbors import KNeighborsClassifier

modelo = KNeighborsClassifier(n_neighbors=3)


## 4.4 Treinar o modelo


In [25]:
modelo.fit(X_treino, y_treino)


KNeighborsClassifier(n_neighbors=3)

O método `fit()` utiliza os dados de treino.

No KNN, o treinamento consiste principalmente em armazenar as observações de treino para posteriormente comparar novas observações com seus vizinhos.


# 5. Avaliação

Agora vamos utilizar os 30 municípios reservados para teste.

Esses municípios não foram utilizados pelo modelo durante o treinamento.


In [26]:
y_pred = modelo.predict(X_teste)

y_pred


array(['Média', 'Média', 'Média', 'Média', 'Grande', 'Pequena', 'Grande',
       'Grande', 'Média', 'Pequena', 'Grande', 'Média', 'Pequena',
       'Grande', 'Pequena', 'Pequena', 'Grande', 'Grande', 'Pequena',
       'Pequena', 'Grande', 'Grande', 'Média', 'Média', 'Pequena',
       'Média', 'Grande', 'Pequena', 'Pequena', 'Média'], dtype=object)

## 5.1 Comparar valores reais e previstos


In [27]:
resultado = pd.DataFrame({
    "populacao_urbana": X_teste["populacao_urbana"],
    "Real": y_teste,
    "Previsto": y_pred
}).sort_values("populacao_urbana")

resultado


,populacao_urbana,Real,Previsto
1,1216,Pequena,Pequena
3,1352,Pequena,Pequena
5,1712,Pequena,Pequena
7,2225,Pequena,Pequena
9,2407,Pequena,Pequena
13,2646,Pequena,Pequena
16,2929,Pequena,Pequena
17,2963,Pequena,Pequena
21,4139,Pequena,Pequena
31,6736,Pequena,Pequena


## 5.2 Calcular a acurácia


In [28]:
from sklearn.metrics import accuracy_score

acuracia = accuracy_score(y_teste, y_pred)

print(f"Acurácia: {acuracia:.2%}")


Acurácia: 100.00%


## 5.3 Matriz de confusão


In [29]:
from sklearn.metrics import confusion_matrix

matriz = confusion_matrix(
    y_teste,
    y_pred,
    labels=["Pequena", "Média", "Grande"]
)

pd.DataFrame(
    matriz,
    index=["Real Pequena", "Real Média", "Real Grande"],
    columns=["Prev. Pequena", "Prev. Média", "Prev. Grande"]
)


,Prev. Pequena,Prev. Média,Prev. Grande
Real Pequena,10,0,0
Real Média,0,10,0
Real Grande,0,0,10


### Atividade 4

Responda:

1. Quantos municípios foram utilizados para treinamento?
2. Quantos foram utilizados para teste?
3. Quantos municípios foram classificados corretamente?
4. Qual foi a acurácia?
5. Em qual classe ocorreram mais erros?
6. Por que usar 100 observações é mais adequado que utilizar apenas 10?
7. O que aconteceria se alterássemos `K=3` para `K=5` ou `K=7`?


# 6. Implantação

Vamos simular uma pequena função que recebe a população urbana de uma nova cidade e devolve a classe prevista pelo modelo.


In [30]:
def prever_porte(populacao_urbana):
    nova_cidade = pd.DataFrame({
        "populacao_urbana": [populacao_urbana]
    })

    return modelo.predict(nova_cidade)[0]


In [31]:
prever_porte(250000)


'Grande'

## 6.1 Teste com diferentes populações

Experimente diferentes valores e observe a classificação retornada pelo modelo.


In [32]:
for populacao in [20000, 100000, 250000, 600000, 1000000]:
    print(
        f"{populacao:>10,} habitantes -> "
        f"{prever_porte(populacao)}"
    )


    20,000 habitantes -> Média
   100,000 habitantes -> Grande
   250,000 habitantes -> Grande
   600,000 habitantes -> Grande
 1,000,000 habitantes -> Grande


# 7. Fechamento — CRISP-DM

Neste notebook percorremos o ciclo completo:

### 1. Compreensão do Negócio
Definimos o problema de classificação dos municípios.

### 2. Compreensão dos Dados
Investigamos a API REST do IBGE e a estrutura JSON.

### 3. Preparação dos Dados
Transformamos o JSON em `DataFrame`, selecionamos 100 municípios e criamos as classes.

### 4. Modelagem
Criamos e treinamos um modelo KNN.

### 5. Avaliação
Utilizamos dados de teste, calculamos acurácia e matriz de confusão.

### 6. Implantação
Criamos uma função para classificar uma nova observação.

O ciclo pode ser repetido posteriormente utilizando outras variáveis, outros estados, novos critérios de classificação ou outros algoritmos.
